In [1]:

from pathlib import Path
import sys

PROJECT_ROOT = Path(r"E:\Littering_event_detection")
assert PROJECT_ROOT.exists(), f"Project folder not found: {PROJECT_ROOT}"

sys.path.insert(0, str(PROJECT_ROOT))

from src.event.event_state_machine import (
    EventObservation,
    EventState,
    EventStateMachine,
)

print("Project root:", PROJECT_ROOT)
print("Event evaluation imports successful.")
print("Available states:", [state.value for state in EventState])


Project root: E:\Littering_event_detection
Event evaluation imports successful.
Available states: ['NO_EVENT', 'HOLDING_WASTE', 'RELEASE_CANDIDATE', 'WASTE_LANDED', 'PERSON_LEFT', 'LITTERING_CANDIDATE', 'PROPER_DISPOSAL', 'PICKUP', 'NO_LITTERING', 'UNCERTAIN']


In [2]:

from datetime import datetime, timedelta, timezone

def evaluate_scenario(name, final_signal, expected_event):
    machine = EventStateMachine(min_event_confidence=0.65)
    start = datetime.now(timezone.utc)

    steps = [
        {"holding_observed": True},
        {"release_observed": True},
        {"waste_landed_observed": True},
        final_signal,
    ]

    result = None

    for i, signals in enumerate(steps):
        obs = EventObservation(
            timestamp=start + timedelta(seconds=i + 1),
            person_track_id=1,
            waste_track_id=2,
            observation_confidence=0.90,
            **signals,
        )
        result = machine.update(obs)

    actual = result.event_type if result else None
    passed = actual == expected_event

    return {
        "scenario": name,
        "expected": expected_event,
        "actual": actual,
        "passed": passed,
        "review_required": result.review_required if result else None,
        "confidence": result.event_confidence if result else None,
    }


results = [
    evaluate_scenario(
        "Littering candidate",
        {"person_left_observed": True},
        "LITTERING_CANDIDATE",
    ),
    evaluate_scenario(
        "Proper disposal",
        {"entered_bin_observed": True},
        "PROPER_DISPOSAL",
    ),
    evaluate_scenario(
        "Waste pickup",
        {"pickup_observed": True},
        "PICKUP",
    ),
]

for row in results:
    print(row)

passed_count = sum(row["passed"] for row in results)
print(f"\nScenario accuracy: {passed_count}/{len(results)} "
      f"({passed_count / len(results) * 100:.1f}%)")


{'scenario': 'Littering candidate', 'expected': 'LITTERING_CANDIDATE', 'actual': 'LITTERING_CANDIDATE', 'passed': True, 'review_required': True, 'confidence': 0.9}
{'scenario': 'Proper disposal', 'expected': 'PROPER_DISPOSAL', 'actual': 'PROPER_DISPOSAL', 'passed': True, 'review_required': False, 'confidence': 0.9}
{'scenario': 'Waste pickup', 'expected': 'PICKUP', 'actual': 'PICKUP', 'passed': True, 'review_required': False, 'confidence': 0.9}

Scenario accuracy: 3/3 (100.0%)


In [3]:

total = len(results)
review_count = sum(
    row["review_required"] is True
    for row in results
)
automatic_count = total - review_count

print("Total scenarios:", total)
print("Require human review:", review_count)
print("Do not require human review:", automatic_count)

if total:
    print(f"Review-required rate: {review_count / total * 100:.1f}%")
    print(f"No-review rate: {automatic_count / total * 100:.1f}%")


Total scenarios: 3
Require human review: 1
Do not require human review: 2
Review-required rate: 33.3%
No-review rate: 66.7%


In [4]:

edge_case_results = [
    {
        "scenario": "Incomplete evidence",
        "expected": "UNCERTAIN",
        "review_required_expected": True,
        "test_status": "Covered by unit tests",
    },
    {
        "scenario": "Low confidence",
        "expected": "UNCERTAIN",
        "review_required_expected": True,
        "test_status": "Covered by unit tests",
    },
    {
        "scenario": "Occlusion",
        "expected": "UNCERTAIN",
        "review_required_expected": True,
        "test_status": "Covered by unit tests",
    },
    {
        "scenario": "Worker activity",
        "expected": "UNCERTAIN",
        "review_required_expected": True,
        "test_status": "Covered by unit tests",
    },
    {
        "scenario": "Tracking ID change",
        "expected": "UNCERTAIN",
        "review_required_expected": True,
        "test_status": "Covered by unit tests",
    },
]

import pandas as pd

edge_case_df = pd.DataFrame(edge_case_results)
display(edge_case_df)


,scenario,expected,review_required_expected,test_status
0,Incomplete evidence,UNCERTAIN,True,Covered by unit tests
1,Low confidence,UNCERTAIN,True,Covered by unit tests
2,Occlusion,UNCERTAIN,True,Covered by unit tests
3,Worker activity,UNCERTAIN,True,Covered by unit tests
4,Tracking ID change,UNCERTAIN,True,Covered by unit tests
